# 실행 제한

실행 제한을 설정하여 Harness 에이전트가 호출 한 번에 수행할 수 있는 작업량을 제어합니다.

| 정보 | 세부 내용 |
|---|---|
| 튜토리얼 | 실행 제한 - maxIterations, timeoutSeconds, maxTokens |
| SDK | boto3 |
| 모델 | Claude Haiku 4.5 (Bedrock) |

`invoke_harness` API는 세 가지 제한 파라미터를 받습니다.

| 파라미터 | 제어 대상 |
|---|---|
| `maxIterations` | 에이전트 루프의 최대 반복 횟수(생각 → 행동 → 관찰 주기) |
| `timeoutSeconds` | 전체 호출에 적용되는 실제 경과 시간 제한 |
| `maxTokens` | 호출 한 번에 모델이 생성할 수 있는 최대 토큰 수 |

이 노트북에서는 각 제한을 살펴보고 에이전트가 제한에 도달하면 어떤 일이 발생하는지 확인합니다.

### 1. 종속성 설치

In [ ]:
!uv pip install -qU -r ../../requirements.txt

계속하기 전에 Jupyter 커널을 다시 시작하세요.

### 2. 설정

In [ ]:
import sys
import time
import uuid
from pathlib import Path
import boto3

# 헬퍼
sys.path.insert(0, str(Path.cwd().parent.parent))

# --- 설정 ---
from helper.iam import create_harness_role, delete_harness_role
from helper.client import get_agentcore_control_client, get_agentcore_client

# --- boto3 클라이언트 생성 ---
control = get_agentcore_control_client()
client = get_agentcore_client()

account_id = boto3.client("sts").get_caller_identity()["Account"]
print(f"Account: {account_id}")

### 3. IAM 역할 및 Harness 생성

In [ ]:
role_arn = create_harness_role()
print(f"\nExecution Role ARN: {role_arn}")

print("Waiting for IAM role to propagate...")
time.sleep(10)
print("Ready!")

In [ ]:
HARNESS_NAME = f"ExecLimits_{uuid.uuid4().hex[:8]}"

resp = control.create_harness(harnessName=HARNESS_NAME, executionRoleArn=role_arn)
harness = resp["harness"]
harness_id = harness["harnessId"]
harness_arn = harness["arn"]
print(f"Harness ID: {harness_id}")

for i in range(12):
    status = control.get_harness(harnessId=harness_id)["harness"]["status"]
    print(f"  {status}")
    if status == "READY":
        print("\u2705 Harness is ready")
        break
    time.sleep(5)

### 헬퍼 - Harness 응답 스트리밍 및 출력

In [ ]:
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"


def invoke(prompt: str, **limits) -> str:
    """프롬프트와 선택적 실행 제한으로 Harness를 호출합니다.

    maxIterations=, timeoutSeconds=, maxTokens= 중 필요한 값을 전달합니다.
    run()으로 VM을 점검할 수 있도록 세션 ID를 반환합니다.
    """
    sid = str(uuid.uuid4()).upper()
    limit_str = ", ".join(f"{k}={v}" for k, v in limits.items()) or "(defaults)"
    print(f"--- Limits: {limit_str} ---")
    print(f"Session: {sid}\n")

    response = client.invoke_harness(
        harnessArn=harness_arn,
        runtimeSessionId=sid,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        model={"bedrockModelConfig": {"modelId": MODEL_ID}},
        **limits,
    )

    for event in response["stream"]:
        if "contentBlockStart" in event:
            start = event["contentBlockStart"].get("start", {})
            if "toolUse" in start:
                print(f"\n[Tool: {start['toolUse'].get('name', '?')}]", flush=True)
        elif "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                print(delta["text"], end="", flush=True)
        elif "messageStop" in event:
            stop = event["messageStop"]
            reason = stop.get("stopReason", "")
            print(f"\n\n\u2192 stopReason: {reason}")
        elif "metadata" in event:
            meta = event["metadata"]
            usage = meta.get("usage", {})
            if usage:
                print(f"\u2192 usage: input={usage.get('inputTokens', 0)}, output={usage.get('outputTokens', 0)}")
        elif "internalServerException" in event:
            print(f"\nError: {event['internalServerException']}")
    print()
    return sid


def run(cmd: str, session_id: str):
    """지정한 세션의 에이전트 VM에서 셸 명령을 실행합니다."""
    print(f"$ {cmd}")
    resp = client.invoke_agent_runtime_command(
        agentRuntimeArn=harness_arn,
        runtimeSessionId=session_id,
        body={"command": cmd},
    )
    for event in resp["stream"]:
        if "chunk" in event:
            chunk = event["chunk"]
            if "contentDelta" in chunk:
                d = chunk["contentDelta"]
                if "stdout" in d:
                    print(d["stdout"], end="", flush=True)
                if "stderr" in d:
                    print(d["stderr"], end="", flush=True)
            elif "contentStop" in chunk:
                print(f"\n[exit: {chunk['contentStop']['exitCode']}]")
    print()

### 4. 최대 반복 횟수 제한

`maxIterations`는 에이전트가 수행할 수 있는 생각 → 행동 → 관찰 루프의 횟수를 제한합니다. `1`로 설정하면 에이전트는 응답하기 전에 도구를 한 번만 호출할 수 있습니다.

빠르게 제한된 답변을 얻고 싶거나 에이전트가 여러 단계의 작업을 수행하지 않게 하려는 경우 유용합니다.

In [ ]:
invoke(
    "Create 3 files: hello.txt, world.txt, and readme.md with some content in each.",
    maxIterations=1,
)

In [ ]:
# 이전 호출의 VM 점검 = 파일 없음
run("ls -la", sid)

In [ ]:
# 비교: 같은 프롬프트에 더 많은 반복 허용
sid = invoke(
    "Create 3 files: hello.txt, world.txt, and readme.md with some content in each.",
    maxIterations=10,
)

In [ ]:
# 이전 호출의 VM 점검 = 파일 3개
run("ls -la", sid)
run("cat *.md", sid)

### 5. 제한 시간 설정

`timeoutSeconds`는 실제 경과 시간 기준의 종료 시점을 설정합니다. 시간이 만료될 때 에이전트가 아직 작업 중이면 호출이 중지됩니다.

프로덕션 환경에서 지연 시간을 예측 가능하게 유지하거나 작업이 끝없이 실행되는 것을 방지할 때 유용합니다.

In [ ]:
# 에이전트에 복잡한 작업을 주되 5초만 허용
invoke(
    "Write a Python script that generates the first 50 prime numbers, save it to primes.py, then run it and show the output.",
    timeoutSeconds=5,
)

In [ ]:
# 같은 작업에 넉넉한 제한 시간 적용
invoke(
    "Write a Python script that generates the first 50 prime numbers, save it to primes.py, then run it and show the output.",
    timeoutSeconds=120,
)

### 6. 최대 토큰 수 제한

`maxTokens`는 모델이 생성할 수 있는 전체 토큰 수를 제한합니다. 비용을 제어하거나 응답을 간결하게 유지할 때 유용합니다.

In [ ]:
# 매우 적은 토큰 예산 - 에이전트 응답이 도중에 중단됨
invoke(
    "Explain the history of the Python programming language in detail.",
    maxTokens=10,
)

In [ ]:
# 더 넉넉한 토큰 예산
invoke(
    "Explain the history of the Python programming language in detail.",
    maxTokens=2048,
)

### 7. 제한 조합

세 가지 제한을 모두 함께 사용할 수 있습니다. 에이전트는 어느 제한이든 도달하는 즉시 중지됩니다.

In [ ]:
invoke(
    "List all files in the current directory, then create a summary.txt with what you found.",
    maxIterations=3,
    timeoutSeconds=30,
    maxTokens=1024,
)

### 8. 리소스 정리

In [ ]:
control.delete_harness(harnessId=harness_id)
print(f"Deleted harness: {harness_id}")

In [ ]:
delete_harness_role()